In [1]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.naive_bayes import BernoulliNB, ComplementNB
from collections import Counter
import pandas as pd
import numpy as np
import h5py
from tqdm import tqdm
import os

root = "."

In [2]:
# Load smiles
data = pd.read_csv(os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL.tsv"), sep='\t')
data['index'] = data.index
ind_to_id = {i: j for i,j in data[['index', 'id']].values}

# Define some paths
PATH_TO_DOCKING_RESULTS_REAL = os.path.join(root, "..", "processed", "unidock_REAL_docking", 'docking_results')
PATH_TO_INPUT_LIGANDS = os.path.join(root, "..", "processed", "unidock_REAL_docking", 'input_ligands')

# Load pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))

def u64x32_to_fp2048(fp_u64_words: np.ndarray) -> np.ndarray:
    """
    fp_u64_words: shape (32,), dtype uint64
    returns: shape (2048,), dtype uint8 with values {0,1}
    """
    fp_u64_words = np.asarray(fp_u64_words, dtype=np.uint64)
    bits = np.unpackbits(fp_u64_words.view(np.uint8), bitorder="little")
    return bits.astype(np.uint8)

In [3]:
# Load ECFPs
with h5py.File(os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL_ECFP4.h5")) as h5:
    ids_ind = h5['ids'][:]
    fps = h5['fps'][:]
    # popc = h5['popc'][:10]
id_to_fp = {ind_to_id[i]:j for i,j in zip(ids_ind, fps)}

In [5]:
DOCKING_RESULTS_REAL = {}
DOCKING_RESULTS_REAL_BACKGROUND = {}

# For each pocket
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS_REAL))):
    # try:
    lines = open(os.path.join(PATH_TO_INPUT_LIGANDS, f"input_ligands_{pocket}.txt"), "r").readlines()
    lines = [i.strip().replace(".sdf", "").split("/")[-1] for i in lines]
    actives = set(lines[:100000])
    inactives = set(lines[100000:])
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'report.csv'))
    scores['set'] = ["inactive" if i in inactives else "active" for i in scores['compound']]
    scores_actives = scores[scores['set'] == 'active'].reset_index(drop=True)
    scores_inactives = scores[scores['set'] == 'inactive'].reset_index(drop=True)
    DOCKING_RESULTS_REAL[pocket] = {i: j for i, j in zip(scores_actives['compound'], scores_actives['score'])}
    DOCKING_RESULTS_REAL_BACKGROUND[pocket] = {i: j for i, j in zip(scores_inactives['compound'], scores_inactives['score'])}
    # except:
    #     pass

100%|██████████| 276/276 [00:31<00:00,  8.83it/s]


In [6]:
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS_REAL))):

    docking_results = []
    for act in sorted(DOCKING_RESULTS_REAL[pocket]):
        docking_results.append([act, DOCKING_RESULTS_REAL[pocket][act]])
    for back in sorted(DOCKING_RESULTS_REAL_BACKGROUND[pocket]):
        docking_results.append([back, DOCKING_RESULTS_REAL_BACKGROUND[pocket][back]])
    docking_results = pd.DataFrame(docking_results, columns=['id', 'score']).sort_values('score').reset_index(drop=True)
    number_actives = 1130
    docking_results['Y'] = [1] * number_actives + [0] * (len(docking_results) - number_actives)
    compounds = np.array(docking_results['id'].tolist())
    Y = np.array(docking_results['Y'].tolist())
    X = np.array([u64x32_to_fp2048(id_to_fp[i]) for i in compounds])
    perm = np.random.permutation(len(X))
    X, Y, compounds = X[perm], Y[perm], compounds[perm]  

    break

  0%|          | 0/276 [00:00<?, ?it/s]


In [12]:
Counter(X.flatten())

Counter({0: 223562494, 1: 7775490})